# Entropy Maps — Full Research Run (from preprocessed data)

Clean, top-to-bottom execution of the complete experimental pipeline, starting from the
**preprocessed** datasets hosted on Kaggle (`fcomovaz/entropy-defects-datasets`).
The raw → preprocessed step is intentionally skipped (it was done offline once).

**Scope (full research run):**

| Stage | What runs | Datasets |
|---|---|---|
| 0 | Environment test (real CUDA) + unit tests | — |
| 1 | Pixel CSV export + round-trip verification | KolektorSDD2, MVTec-AD (normal + inverted) |
| 2 | Multiscale entropy maps $\mathbf{x}_{85}$ | both, normal + inverted |
| 3 | Spatial asymmetry features $\mathbf{x}_{85}\rightarrow\mathbf{x}_{94}$ | all variants |
| 4 | Classic ML (`dt`, `knn`, `lr`, `rf`) | KolektorSDD2 + **all 15 MVTec-AD objects** |
| 5 | Lightweight CNNs (7 architectures) | KolektorSDD2 + **all 15 MVTec-AD objects** |
| 6 | 100 % CPU fair benchmark (Phase A) | KolektorSDD2 + all MVTec-AD objects |
| 7 | Granular CPU profiling (Phase B) | KolektorSDD2 + all MVTec-AD objects |

**How to use:**
1. `Runtime → Change runtime type → GPU` (stage 0 fails without real CUDA; stage 5 trains on GPU).
2. Edit the **Run configuration** cell if needed.
3. `Runtime → Run all`. Every command is printed before it runs; any non-zero exit stops the notebook.

Every executed command, its duration and exit code are appended to
`results/colab_run_manifest.json` together with the git commit and hardware info.

> Timing results from stages 6–7 describe the Colab CPU recorded in the manifest, not the
> local workstation nor embedded hardware.

# Run configuration

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/fcomovaz/entropy-maps-symmetry2026.git"
REPO_PATH = Path("/content/entropy-maps-symmetry2026")
GIT_REF = None  # e.g. "65e668f" to pin an exact commit; None = default branch HEAD

KAGGLE_DATASET = "fcomovaz/entropy-defects-datasets"

# Regenerate every derived artifact from the preprocessed images (recommended for a clean run).
OVERWRITE = True
# Run the full unit-test suite in stage 0.
RUN_UNIT_TESTS = True

# Long runs: copy results/ to Google Drive after every stage so a disconnect loses nothing.
SAVE_TO_DRIVE = False
DRIVE_DIR = Path("/content/drive/MyDrive/entropy-maps-runs")

# General configurations
## Getting the repo from GitHub

In [ ]:
import subprocess

if not REPO_PATH.exists():
    !git clone {REPO_URL} {REPO_PATH}
elif not (REPO_PATH / ".git").exists():
    print("⚠️  Folder exists but is not a valid repo. Re-cloning...")
    !rm -rf {REPO_PATH}
    !git clone {REPO_URL} {REPO_PATH}
else:
    print(f"✅ Existing repo found at {REPO_PATH}")

%cd {REPO_PATH}

if GIT_REF:
    !git fetch --all --quiet
    !git checkout --quiet {GIT_REF}

GIT_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_PATH, capture_output=True, text=True, check=True
).stdout.strip()
print("Commit:", GIT_COMMIT)

## Installing the requirements
Colab already ships PyTorch, NumPy, pandas, scikit-learn, etc. Only these are missing on Linux/Colab.

In [ ]:
%pip install -q torchinfo psutil kagglehub

## Command runner
`run_stage` executes each command as a subprocess from the repo root, streams its output,
**stops on the first failure**, and logs everything to `results/colab_run_manifest.json`.

In [ ]:
import json
import os
import platform
import shlex
import shutil
import sys
import time
from datetime import datetime, timezone

import torch

MANIFEST_PATH = REPO_PATH / "results" / "colab_run_manifest.json"


def _cpu_model() -> str:
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"


def _load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "git_commit": GIT_COMMIT,
        "overwrite": OVERWRITE,
        "environment": {
            "python": sys.version.split()[0],
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cpu_model": _cpu_model(),
            "cpu_count": os.cpu_count(),
            "platform": platform.platform(),
        },
        "commands": [],
    }


def _save_manifest(manifest: dict) -> None:
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))


def _sync_to_drive() -> None:
    if not SAVE_TO_DRIVE:
        return
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    dest = DRIVE_DIR / GIT_COMMIT[:7] / "results"
    shutil.copytree(REPO_PATH / "results", dest, dirs_exist_ok=True)
    print(f"☁️  results/ synced to {dest}")


def ow(args: list[str]) -> list[str]:
    """Append --overwrite when OVERWRITE is enabled."""
    return args + ["--overwrite"] if OVERWRITE else args


def run_stage(stage: str, commands: list[list[str]]) -> None:
    """Run each command (args after `python`) in order; abort on the first non-zero exit."""
    manifest = _load_manifest()
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    stage_t0 = time.perf_counter()
    print(f"=== {stage} ({len(commands)} commands) ===")
    for args in commands:
        cmd = [sys.executable, *args]
        preview = "python " + shlex.join(args)
        print(f"\n▶ {preview}", flush=True)
        t0 = time.perf_counter()
        started = datetime.now(timezone.utc).isoformat()
        proc = subprocess.Popen(
            cmd, cwd=REPO_PATH, env=env, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        returncode = proc.wait()
        elapsed = time.perf_counter() - t0
        manifest["commands"].append({
            "stage": stage, "command": preview, "started_utc": started,
            "duration_s": round(elapsed, 2), "returncode": returncode,
        })
        _save_manifest(manifest)
        if returncode != 0:
            raise RuntimeError(f"[{stage}] failed (exit {returncode}): {preview}")
        print(f"✔ done in {elapsed:.1f} s")
    print(f"\n✅ {stage} finished in {(time.perf_counter() - stage_t0) / 60:.2f} min")
    _sync_to_drive()

# Stage 0 — Environment and tests
`test_environment.py` requires real CUDA execution and fails otherwise.

In [ ]:
stage0 = [["tests/test_environment.py"]]
if RUN_UNIT_TESTS:
    stage0.append(["-m", "unittest", "discover", "tests", "-v"])
run_stage("0_environment_tests", stage0)

# Getting the preprocessed datasets from Kaggle
All images are 128×128 grayscale. KolektorSDD2 (rectangular) was letterboxed with zero padding.
Each dataset also has an `-inverted` variant ($255 - I$, applied before letterboxing).

In [ ]:
import kagglehub

data_dir = REPO_PATH / "data"
preprocessed_dir = data_dir / "preprocessed"
REQUIRED = ["mvtec-ad", "mvtec-ad-inverted", "KolektorSDD2", "KolektorSDD2-inverted"]


def _png_count(path: Path) -> int:
    return sum(1 for _ in path.rglob("*.png")) if path.exists() else 0


if all(_png_count(preprocessed_dir / name) > 0 for name in REQUIRED):
    print(f"✅ Preprocessed datasets already present in {preprocessed_dir}")
else:
    print("⏳ Downloading preprocessed datasets from Kaggle...")
    cache = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print(f"✅ Cached at: {cache}")
    data_dir.mkdir(parents=True, exist_ok=True)
    res = subprocess.run(["cp", "-a", f"{cache}/.", str(data_dir)], capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
    print("📂 Copied into data/")

for name in REQUIRED:
    n = _png_count(preprocessed_dir / name)
    print(f"  {name:24s} {n:>6} PNG files")
    if n == 0:
        raise FileNotFoundError(f"Missing or empty: {preprocessed_dir / name}")

# Stage 1 — Pixel CSV export
Each 128×128 image becomes one row (`px_00000 … px_16383` + metadata).
The `understanding/09` and `understanding/11` scripts verify the bit-exact round trip CSV → image.

In [ ]:
run_stage("1_csv_export", [
    ow(["-m", "experiments.export_mvtec_ad_csv"]),
    ow(["-m", "experiments.export_mvtec_ad_csv", "--inverted"]),
    ow(["-m", "experiments.export_kolektorsdd2_csv"]),
    ow(["-m", "experiments.export_kolektorsdd2_csv", "--inverted"]),
    ["understanding/09_verify_mvtec_csv/verify.py"],
    ["understanding/09_verify_mvtec_csv/verify.py", "--inverted"],
    ["understanding/11_verify_kolektorsdd2_csv/verify.py"],
    ["understanding/11_verify_kolektorsdd2_csv/verify.py", "--inverted"],
])

# Stage 2 — Multiscale entropy maps
Grid partitions $g \in \{1,2,4,8\}$:

$$\mathbf{x}_{85}= \left[ \underbrace{H^{(1)},H^{(2)},H^{(4)},H^{(8)}}_{1+4+16+64=85\text{ features}}\right]$$

In [ ]:
run_stage("2_entropy_maps", [
    ow(["-m", "experiments.export_entropy_maps", "--config", "configs/entropy/mvtec_ad.toml"]),
    ow(["-m", "experiments.export_entropy_maps", "--config", "configs/entropy/mvtec_ad.toml", "--inverted"]),
    ow(["-m", "experiments.export_entropy_maps", "--config", "configs/entropy/kolektorsdd2.toml"]),
    ow(["-m", "experiments.export_entropy_maps", "--config", "configs/entropy/kolektorsdd2.toml", "--inverted"]),
])

# Stage 3 — Spatial asymmetry features
Appends 9 multiscale bilateral/rotational entropy residuals: $\mathbf{x}_{94} = [\mathbf{x}_{85}, \mathbf{x}_{9}]$.
`--all` processes every config in `configs/asymmetry/` for the normal and inverted variants.

In [ ]:
run_stage("3_asymmetry_features", [
    ow(["-m", "experiments.append_asymmetry_features", "--all"]),
])

# Stage 4 — Classic ML on $\mathbf{x}_{94}$
All classifiers (`dt`, `knn`, `lr`, `rf`), subset ablation ($\mathbf{x}_{85}$ / $\mathbf{x}_{9}$ / $\mathbf{x}_{94}$) and spatial explainability.
MVTec-AD uses an in-memory stratified split per object (its native `train` has only `good` samples).

In [ ]:
run_stage("4_classic_ml", [
    ow(["-m", "experiments.evaluate_descriptors", "--config", "configs/classification/kolektorsdd2.toml", "--classifier", "all"]),
    ow(["-m", "experiments.evaluate_descriptors", "--config", "configs/classification/kolektorsdd2.toml", "--classifier", "all", "--inverted"]),
    ow(["-m", "experiments.evaluate_descriptors", "--config", "configs/classification/mvtec_ad.toml", "--all-objects", "--classifier", "all"]),
    ow(["-m", "experiments.evaluate_descriptors", "--config", "configs/classification/mvtec_ad.toml", "--all-objects", "--classifier", "all", "--inverted"]),
])

# Stage 5 — Lightweight CNNs
7 architectures (`ultra_tiny`, `ds_cnn`, `tiny_cnn`, `mobilenet_like`, `tiny_resnet`, `small_cnn`, `torchvision_mobilenet`)
with the seeds and threshold calibration defined in `configs/cnn/*.toml`. This is the longest stage.
The checkpoints produced here are required by stages 6 and 7.

In [ ]:
run_stage("5_cnn", [
    ow(["-m", "experiments.evaluate_cnn", "--config", "configs/cnn/kolektorsdd2.toml", "--model", "all"]),
    ow(["-m", "experiments.evaluate_cnn", "--config", "configs/cnn/kolektorsdd2.toml", "--model", "all", "--inverted"]),
    ow(["-m", "experiments.evaluate_cnn", "--config", "configs/cnn/mvtec_ad.toml", "--all-objects", "--model", "all"]),
    ow(["-m", "experiments.evaluate_cnn", "--config", "configs/cnn/mvtec_ad.toml", "--all-objects", "--model", "all", "--inverted"]),
])

# Stage 6 — 100 % CPU fair benchmark (Phase A)
Descriptor + classifier vs CNNs, everything on CPU, reporting
$T_{\text{total}} = T_{\text{descriptor}} + T_{\text{model}}$, parameters, FLOPs and throughput.
Consolidated table: `results/comparison_cpu/summary_all_datasets.csv`.

In [ ]:
run_stage("6_cpu_fair_benchmark", [
    ["-m", "experiments.benchmark_cpu_fair", "--config", "configs/profiling/kolektorsdd2.toml"],
    ["-m", "experiments.benchmark_cpu_fair", "--config", "configs/profiling/kolektorsdd2.toml", "--inverted"],
    ["-m", "experiments.benchmark_cpu_fair", "--config", "configs/profiling/mvtec_ad.toml", "--all-objects"],
    ["-m", "experiments.benchmark_cpu_fair", "--config", "configs/profiling/mvtec_ad.toml", "--all-objects", "--inverted"],
])

# Stage 7 — Granular CPU profiling (Phase B)
Per-stage descriptor timing and per-layer CNN profiling. With `--inverted`, the config runs only the short
$\mathbf{x}_{94}$ descriptor check (no redundant full CNN profiling).

In [ ]:
run_stage("7_cpu_profiling", [
    ["-m", "experiments.profile_pipelines", "--config", "configs/profiling/kolektorsdd2.toml"],
    ["-m", "experiments.profile_pipelines", "--config", "configs/profiling/kolektorsdd2.toml", "--inverted"],
    ["-m", "experiments.profile_pipelines", "--config", "configs/profiling/mvtec_ad.toml", "--all-objects"],
    ["-m", "experiments.profile_pipelines", "--config", "configs/profiling/mvtec_ad.toml", "--all-objects", "--inverted"],
])

# Run summary

In [ ]:
import pandas as pd

manifest = json.loads(MANIFEST_PATH.read_text())
log = pd.DataFrame(manifest["commands"])
print("Commit:", manifest["git_commit"])
print("Environment:", json.dumps(manifest["environment"], indent=2))
display(log.groupby("stage", sort=False)["duration_s"].agg(["count", "sum"]).rename(columns={"sum": "total_s"}))

summary_csv = REPO_PATH / "results" / "comparison_cpu" / "summary_all_datasets.csv"
if summary_csv.exists():
    display(pd.read_csv(summary_csv))
else:
    print(f"⚠️  {summary_csv} not found")

# Save results
Zips `results/` plus the descriptor CSVs (`*-entropy*.csv/.json`, not the large pixel CSVs) and downloads it.

In [ ]:
import zipfile

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path(f"/content/entropy_maps_results_{GIT_COMMIT[:7]}_{stamp}.zip")
to_pack = [p for p in (REPO_PATH / "results").rglob("*") if p.is_file()]
to_pack += [p for p in (REPO_PATH / "data" / "csv").glob("*-entropy*") if p.is_file()]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in to_pack:
        zf.write(p, p.relative_to(REPO_PATH))
print(f"📦 {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB, {len(to_pack)} files)")

_sync_to_drive()
if SAVE_TO_DRIVE:
    shutil.copy2(zip_path, DRIVE_DIR / zip_path.name)

from google.colab import files
files.download(str(zip_path))